In [1]:
import sys
from pathlib import Path

# Repo-root notebook: put src/ on path if the package isn't installed editable.
_src = Path.cwd() / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datalake.gateway import DataLakeGateway
from domain.ports.data_catalog import DEFAULT_DATA_ROOT

GATEWAY = DataLakeGateway(root=DEFAULT_DATA_ROOT)


def load_session(symbol: str, trade_date: str | None = None, timeframe: str = "5m") -> pd.DataFrame:
    """Load one session of OHLCV from the local Parquet lake (`data/lake`).

    5m timestamps are naive IST wall-clock; `date` is localized to
    Asia/Kolkata (IST). `trade_date` = "YYYY-MM-DD" loads that IST day,
    None = latest session.
    """
    if trade_date:
        # ponytail: gateway filters midnight-bounded (IST afternoon → UTC),
        # so pull a window and keep the exact IST calendar day as source of truth.
        df = GATEWAY.history(symbol, timeframe=timeframe, from_date=trade_date)
    else:
        df = GATEWAY.history(symbol, timeframe=timeframe, lookback_days=1)
    if df.empty:
        return df
    out = df.copy()
    # 5m lake stores naive IST wall-clock (no UTC shift) — localize directly.
    out["date"] = pd.to_datetime(out["timestamp"]).dt.tz_localize("Asia/Kolkata")
    if trade_date:
        out = out[out["date"].dt.date.astype(str) == trade_date]
    # Drop the phantom 00:00 UTC (05:30 IST) pre-open bar the lake prepends
    # each day — it bundles the overnight window, not real trading.
    out = out[out["date"].dt.time >= pd.Timestamp("09:15").time()]
    return out.drop(columns=["timestamp"]).reset_index(drop=True)


def resample_ohlc(df: pd.DataFrame, rule: str = "15min") -> pd.DataFrame:
    """Resample session OHLCV to a coarser rule (e.g. '15min') for indicators."""
    return (
        df.set_index("date")
        .resample(rule)
        .agg(open=("open", "first"), high=("high", "max"),
             low=("low", "min"), close=("close", "last"), volume=("volume", "sum"))
        .dropna(subset=["close"])
        .reset_index()
    )


def compute_obv_sma(df: pd.DataFrame, rule: str = "15min", obv_sma: int = 14) -> pd.DataFrame:
    """OBV on ``rule``-resampled bars, plus its SMA(``obv_sma``) -> OBV_SMA.

    ponytail: OBV is seeded at 0 each session (no cross-session carry) — fine for
    intraday visualization; cross-day continuity would need the full history.
    """
    r = resample_ohlc(df, rule)
    sign = np.sign(r["close"].diff().fillna(0.0))
    r["obv"] = (sign * r["volume"]).cumsum()
    r["obv_sma"] = r["obv"].rolling(obv_sma, min_periods=obv_sma).mean()
    return r


def show_candlestick(df: pd.DataFrame, symbol: str, height: int = 900,
                     obv_sma: int | None = 14, rule: str = "15min"):
    """Render a TradingView-style dark candlestick + volume chart (IST x-axis).

    When ``obv_sma`` is set, a third panel shows OBV and its SMA computed on
    ``rule``-resampled bars (default 15m) — independent of the 5m candles.
    """
    dark_bg, dark_grid, dark_font = "#131722", "#222634", "#B2B5BE"
    dark_axis, up, down = "#44475a", "#26a69a", "#ef5350"

    vol_colors = [up if c >= o else down for o, c in zip(df["open"], df["close"])]

    has_obv = obv_sma is not None
    n_rows = 3 if has_obv else 2
    row_heights = [0.60, 0.18, 0.22] if has_obv else [0.72, 0.28]

    fig = make_subplots(
        rows=n_rows, cols=1, shared_xaxes=True,
        vertical_spacing=0.03, row_heights=row_heights,
    )
    fig.add_trace(
        go.Candlestick(
            x=df["date"], open=df["open"], high=df["high"],
            low=df["low"], close=df["close"], name=symbol,
            increasing=dict(line=dict(color=up), fillcolor=up),
            decreasing=dict(line=dict(color=down), fillcolor=down),
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Bar(x=df["date"], y=df["volume"], name="Volume",
               marker_color=vol_colors, opacity=0.85),
        row=2, col=1,
    )
    axis_kw = dict(
        tickfont=dict(color=dark_font), gridcolor=dark_grid,
        linecolor=dark_axis, zerolinecolor=dark_axis,
    )
    if has_obv:
        obv_df = compute_obv_sma(df, rule=rule, obv_sma=obv_sma)
        fig.add_trace(
            go.Scatter(x=obv_df["date"], y=obv_df["obv"], name="OBV (15m)",
                       line=dict(color="#5b9bd5", width=1.2)),
            row=3, col=1,
        )
        fig.add_trace(
            go.Scatter(x=obv_df["date"], y=obv_df["obv_sma"],
                       name=f"OBV SMA{obv_sma}", line=dict(color="#ed9c28", width=1.5)),
            row=3, col=1,
        )
    fig.update_layout(
        title=dict(text=f"Candlestick Chart for {symbol}", font=dict(color=dark_font)),
        height=height, showlegend=has_obv,
        plot_bgcolor=dark_bg, paper_bgcolor=dark_bg, font=dict(color=dark_font),
        xaxis_rangeslider_visible=False,
    )
    fig.update_xaxes(**axis_kw, row=1, col=1)
    fig.update_yaxes(title=dict(text="Price", font=dict(color=dark_font)), **axis_kw, row=1, col=1)
    fig.update_yaxes(title=dict(text="Volume", font=dict(color=dark_font)), **axis_kw, row=2, col=1)
    if has_obv:
        fig.update_xaxes(**axis_kw, row=2, col=1)
        fig.update_yaxes(title=dict(text="OBV (15m)", font=dict(color=dark_font)), **axis_kw, row=3, col=1)
        fig.update_xaxes(title=dict(text="Date (IST)", font=dict(color=dark_font)), **axis_kw, row=3, col=1)
    else:
        fig.update_xaxes(title=dict(text="Date (IST)", font=dict(color=dark_font)), **axis_kw, row=2, col=1)
    fig.show()


def chart_candles(symbol: str, trade_date: str | None = None, height: int = 900,
                  obv_sma: int | None = 14, rule: str = "15min") -> pd.DataFrame:
    """One-call convenience: load a session then show its candlestick chart.

    chart_candles("RELIANCE")                     # latest session, 5m + OBV(15m)
    chart_candles("RELIANCE", "2026-07-13")       # specific day
    chart_candles("RELIANCE", obv_sma=None)       # hide OBV panel
    chart_candles("RELIANCE", rule="30min", obv_sma=20)  # other resample/SMA
    """
    df = load_session(symbol, trade_date)
    if df.empty:
        print(f"No data for {symbol} ({trade_date or 'latest'})")
        return df
    print(
        f"{symbol}: {len(df)} bars  "
        f"{df['date'].min()} → {df['date'].max()}"
    )
    show_candlestick(df, symbol, height=height, obv_sma=obv_sma, rule=rule)
    return df






20:10:15.506 INFO     infrastructure.logging_config  Logging configured - service=tradexv2 level=INFO format=human log_file=None [no-correlation]


In [2]:
# Pure Morning Momentum scanner (Algo 3) — date → ranked passes → Top-3 sector-diversified picks
from __future__ import annotations

from pathlib import Path

import duckdb
import pandas as pd

from analytics.sector.mapping import SectorMapper
from domain.ports.data_catalog import DEFAULT_DATA_ROOT

# --- user input: set TRADE_DATE, or leave None to use latest session day in the lake ---
TRADE_DATE: str | None = "2026-07-13"  # e.g. "2026-07-13"
TOP_K = 3
MIN_MORNING_GAIN = 0.50
MIN_RVOL = 2.0
MIN_CLOSE_IN_RANGE = 0.80
RVOL_LOOKBACK_DAYS = 14

# Lake stores IST session clock shifted to UTC wall: 09:15 IST → 03:45, 09:45 IST → 04:15
MORNING_START = "03:45:00"
MORNING_END = "04:15:00"

_PARQUET_GLOB = str(
    Path(DEFAULT_DATA_ROOT) / "equities/candles/timeframe=1m/symbol=*/data.parquet"
)


def resolve_trade_date(con: duckdb.DuckDBPyConnection, trade_date: str | None) -> str:
    if trade_date:
        return trade_date
    row = con.execute(
        f"""
        SELECT max(CAST(timestamp AS DATE))::VARCHAR
        FROM read_parquet('{_PARQUET_GLOB}', hive_partitioning := true)
        """
    ).fetchone()
    if not row or not row[0]:
        raise RuntimeError("No candle dates found in datalake")
    return row[0]


def scan_morning_momentum(trade_date: str) -> pd.DataFrame:
    """Build on-the-fly session_0945 features and apply Pure Morning Momentum filters."""
    sql = f"""
    WITH bars AS (
      SELECT
        symbol,
        timestamp,
        CAST(timestamp AS DATE) AS td,
        open, high, low, close, volume
      FROM read_parquet('{_PARQUET_GLOB}', hive_partitioning := true)
      WHERE CAST(timestamp AS DATE)
              BETWEEN DATE '{trade_date}' - INTERVAL 45 DAY AND DATE '{trade_date}'
        AND CAST(timestamp AS TIME) BETWEEN TIME '{MORNING_START}' AND TIME '{MORNING_END}'
    ),
    session AS (
      SELECT
        symbol,
        td,
        arg_min(open, timestamp) AS day_open,
        max(high) AS high_0945,
        min(low) AS low_0945,
        arg_max(close, timestamp) AS c945,
        sum(volume) AS morning_vol,
        count(*) AS morning_bars
      FROM bars
      GROUP BY symbol, td
    ),
    hist AS (
      SELECT *,
        row_number() OVER (PARTITION BY symbol ORDER BY td DESC) AS rn
      FROM session
      WHERE td < DATE '{trade_date}'
    ),
    baseline AS (
      SELECT symbol, avg(morning_vol) AS avg_morning_vol
      FROM hist
      WHERE rn <= {RVOL_LOOKBACK_DAYS}
      GROUP BY symbol
    ),
    today AS (
      SELECT
        s.symbol,
        s.c945,
        s.day_open,
        s.high_0945,
        s.low_0945,
        s.morning_vol,
        s.morning_bars,
        (s.c945 - s.day_open) / NULLIF(s.day_open, 0) * 100 AS morning_gain,
        CASE
          WHEN s.high_0945 > s.low_0945
          THEN (s.c945 - s.low_0945) / (s.high_0945 - s.low_0945)
          ELSE NULL
        END AS close_in_range,
        s.morning_vol / NULLIF(b.avg_morning_vol, 0) AS rvol
      FROM session s
      LEFT JOIN baseline b USING (symbol)
      WHERE s.td = DATE '{trade_date}'
    )
    SELECT *
    FROM today
    WHERE morning_gain > {MIN_MORNING_GAIN}
      AND rvol > {MIN_RVOL}
      AND close_in_range > {MIN_CLOSE_IN_RANGE}
    ORDER BY rvol DESC
    """
    return duckdb.connect().execute(sql).fetchdf()


def industry_diversified_top_k(passes: pd.DataFrame, k: int = TOP_K) -> pd.DataFrame:
    """Walk rvol-ranked list; keep at most one symbol per sector (SectorMapper)."""
    mapper = SectorMapper.default()
    df = mapper.assign_sectors(passes.reset_index(drop=True))
    picked: list[int] = []
    seen_sectors: set[str] = set()
    for i, row in df.iterrows():
        sector = row["sector"]
        if sector in seen_sectors:
            continue
        seen_sectors.add(sector)
        picked.append(i)
        if len(picked) >= k:
            break
    return df.loc[picked].reset_index(drop=True)


con = duckdb.connect()
trade_date = resolve_trade_date(con, TRADE_DATE)
print(f"Scanning Pure Morning Momentum for {trade_date} …")

passes = scan_morning_momentum(trade_date)
picks = industry_diversified_top_k(passes, k=TOP_K)

cols = [
    "symbol",
    "sector",
    "c945",
    "morning_gain",
    "rvol",
    "close_in_range",
    "morning_vol",
    "morning_bars",
]

print(f"Filter passes: {len(passes)}  |  Top-{TOP_K} diversified picks: {len(picks)}")
print("\n=== Top-K picks ===")
display(picks[cols].round(4))
print("\n=== All scanner passes (by rvol) ===")
all_passes = SectorMapper.default().assign_sectors(passes)
display(all_passes[cols].round(4))


Scanning Pure Morning Momentum for 2026-07-13 …
20:10:28.301 INFO     analytics.sector.mapping       SectorMapper loaded 511 symbols from data/sectors/nifty_sector_mapping.csv [no-correlation]
Filter passes: 0  |  Top-3 diversified picks: 0

=== Top-K picks ===


,symbol,sector,c945,morning_gain,rvol,close_in_range,morning_vol,morning_bars



=== All scanner passes (by rvol) ===
20:10:28.338 INFO     analytics.sector.mapping       SectorMapper loaded 511 symbols from data/sectors/nifty_sector_mapping.csv [no-correlation]


,symbol,sector,c945,morning_gain,rvol,close_in_range,morning_vol,morning_bars


In [3]:
# Afternoon Expansion Scanner — picks @09:50 IST from data/lake.
# Thin wrapper over analytics.intraday.afternoon_expansion (kept clean +
# test-covered there). Ranking uses only morning + prior-day history;
# the realized_* columns are backcheck-only (never used for selection).
from __future__ import annotations

from analytics.intraday.afternoon_expansion import (
    AfternoonExpansionConfig,
    RESULT_COLUMNS,
    run_scan,
)

# --- user inputs ---
TRADE_DATE: str | None = "2026-07-17"  # None = latest lake day
TOP_K = 5
MIN_RVOL = 1.5
MIN_ABS_MORNING_GAIN = 0.30
MIN_HIST_SESSIONS = 20
MIN_HIT_GE5 = 0.05
TARGET_MFE_LO, TARGET_MFE_HI = 5.0, 10.0

config = AfternoonExpansionConfig(
    top_k=TOP_K,
    min_rvol=MIN_RVOL,
    min_abs_morning_gain=MIN_ABS_MORNING_GAIN,
    min_hist_sessions=MIN_HIST_SESSIONS,
    min_hit_ge5=MIN_HIT_GE5,
    target_mfe_lo=TARGET_MFE_LO,
    target_mfe_hi=TARGET_MFE_HI,
)

trade_date = TRADE_DATE  # run_scan resolves None internally
print(
    f"Afternoon expansion scan @ 09:50 IST for {trade_date or '<latest>'}\n"
    f"(label = max excursion 09:50->15:15 IST; target band {TARGET_MFE_LO}-{TARGET_MFE_HI}%)"
)

passes, picks = run_scan(trade_date, config=config, pick=True)

print(f"Candidates: {len(passes)}  |  Top-{TOP_K} diversified: {len(picks)}")
print("\n=== Top picks (trade from ~09:50) ===")
display(picks[RESULT_COLUMNS].round(4)) if len(picks) else picks
print("\n=== Full ranked list ===")
display(passes[RESULT_COLUMNS].round(4)) if len(passes) else passes



Afternoon expansion scan @ 09:50 IST for 2026-07-17
(label = max excursion 09:50->15:15 IST; target band 5.0-10.0%)
20:10:34.943 INFO     analytics.sector.mapping       SectorMapper loaded 511 symbols from data/sectors/nifty_sector_mapping.csv [no-correlation]
20:10:34.952 INFO     analytics.sector.mapping       SectorMapper loaded 511 symbols from data/sectors/nifty_sector_mapping.csv [no-correlation]
Candidates: 5  |  Top-5 diversified: 4

=== Top picks (trade from ~09:50) ===


,symbol,sector,c950,morning_gain,rvol,close_in_range,hit_ge5,hit_5_10,avg_mfe,p90_mfe,hist_sessions,realized_mfe_after_0950,realized_range_after_0950
0,AEGISVOPAK,OilGas,270.77,2.2661,1.9572,0.7099,0.1667,0.1429,3.2748,5.7826,42,2.1753,3.5676
1,NEWGEN,IT,528.65,-5.5307,1.8219,0.0388,0.0952,0.0476,2.5468,4.8040,42,5.2587,6.0437
2,COHANCE,Pharma,432.75,-4.5545,5.6550,0.0262,0.0714,0.0476,2.0839,3.9167,42,2.5881,2.9925
3,KALYANKJIL,ConsumerDur,555.15,1.6851,2.6181,0.5419,0.0714,0.0714,2.4601,4.4872,42,4.0259,4.5033



=== Full ranked list ===


,symbol,sector,c950,morning_gain,rvol,close_in_range,hit_ge5,hit_5_10,avg_mfe,p90_mfe,hist_sessions,realized_mfe_after_0950,realized_range_after_0950
0,AEGISVOPAK,OilGas,270.77,2.2661,1.9572,0.7099,0.1667,0.1429,3.2748,5.7826,42,2.1753,3.5676
1,NEWGEN,IT,528.65,-5.5307,1.8219,0.0388,0.0952,0.0476,2.5468,4.8040,42,5.2587,6.0437
2,COHANCE,Pharma,432.75,-4.5545,5.6550,0.0262,0.0714,0.0476,2.0839,3.9167,42,2.5881,2.9925
3,VIJAYA,Pharma,1332.40,-7.1304,3.9935,0.0510,0.0714,0.0714,2.2859,4.1497,42,1.7187,2.7319
4,KALYANKJIL,ConsumerDur,555.15,1.6851,2.6181,0.5419,0.0714,0.0714,2.4601,4.4872,42,4.0259,4.5033


In [4]:
# --- user input ---
SYMBOL = "ZENSARTECH"
TRADE_DATE = "2026-07-15"  # "YYYY-MM-DD" or None = latest session

chart_candles(SYMBOL, TRADE_DATE)

ZENSARTECH: 75 bars  2026-07-15 09:15:00+05:30 → 2026-07-15 15:25:00+05:30


,open,high,low,close,volume,oi,symbol,exchange,timeframe,date
0,526.90,527.90,514.25,517.25,405346.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 09:15:00+05:30
1,517.00,520.30,512.15,519.00,189078.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 09:20:00+05:30
2,518.85,524.70,518.20,523.50,187990.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 09:25:00+05:30
3,523.50,524.80,519.50,521.30,135373.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 09:30:00+05:30
4,521.25,523.55,520.55,521.50,76917.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 09:35:00+05:30
...,...,...,...,...,...,...,...,...,...,...
70,512.60,514.40,512.05,513.50,56298.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 15:05:00+05:30
71,513.50,513.80,511.30,512.15,57001.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 15:10:00+05:30
72,512.15,513.90,511.45,512.80,85767.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 15:15:00+05:30
73,513.20,514.00,512.10,513.50,105882.0,0.0,ZENSARTECH,NSE,5m,2026-07-15 15:20:00+05:30
